# Исследование и подготовка данных

Первый этап проекта: знакомство с датасетом, целевой переменной, распределением классов, текстовой подготовкой, стратифицированным split и выбором длины токенизации

## 1. Environment and configuration

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from collections import Counter
from datasets import load_dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from src.project_utils import (
    SEED, MODEL_NAME, DATASET_NAME, NUM_LABELS, MAX_LENGTH,
    set_seed, clean_html, build_text, build_text_with_attributes,
    compute_metrics, make_model, WeightedTrainer, predict_category,
)

set_seed(SEED)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


c:\Users\alive\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.14.0+cu126
CUDA: True


## 2. Dataset

In [2]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print(dataset["train"].column_names)
print("Rows:", dataset["train"].num_rows)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'category', 'parent_category', 'grandparent_category', 'greatgrandparent_category', 'rating', 'description', 'comments', 'attributes', 'seller', 'seller_description', 'adultCategory'],
        num_rows: 2059468
    })
})
['id', 'title', 'category', 'parent_category', 'grandparent_category', 'greatgrandparent_category', 'rating', 'description', 'comments', 'attributes', 'seller', 'seller_description', 'adultCategory']
Rows: 2059468


In [3]:
target_column = "greatgrandparent_category"
none_count = sum(value is None for value in dataset["train"][target_column])
print("Total:", dataset["train"].num_rows)
print("Missing target:", none_count)
print("Valid target:", dataset["train"].num_rows - none_count)

Total: 2059468
Missing target: 657606
Valid target: 1401862


In [4]:
dataset = dataset.filter(lambda example: example[target_column] is not None)
label_names = sorted(set(dataset["train"][target_column]))
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}
print("Classes:", len(label_names))

Filter: 100%|██████████| 2059468/2059468 [00:18<00:00, 109191.75 examples/s]


Classes: 39


In [5]:
category_counts = Counter(dataset["train"][target_column])
for category, count in category_counts.most_common():
    print(f"{category}: {count}")

Аксессуары: 190676
Одежда: 158793
Красота: 144075
Товары для дома: 136703
Электроника: 134292
Хобби и творчество: 128526
Детские товары: 68724
Строительство и ремонт: 59284
Автотовары: 49958
Детская одежда: 42004
Спорт и отдых: 41740
Смартфоны и телефоны: 39178
Зоотовары: 34145
Товары для кухни: 29004
Бытовая техника: 24803
Виды спорта: 15484
Женские аксессуары: 15367
Товары для праздников: 15210
Обувь: 13361
Книги: 9617
Отдых на природе, рыбалка и охота: 8308
Дача, сад и огород: 7366
Уход за волосами: 6361
Здоровье: 5762
Хозяйственные товары: 3435
Техника для дома: 2920
Продукты питания: 2136
Аксессуары и принадлежности: 2079
Ручной инструмент и оснастка: 1978
Бытовая химия и личная гигиена: 1883
Женская одежда: 1882
Текстиль: 1546
Электроинструменты: 1521
Компьютерная техника: 972
Техника для кухни: 759
Офисная техника: 756
Гигиена и подгузники: 673
Детский спорт и активный отдых: 376
Наушники и аудиотехника: 205


In [9]:
dataset = dataset.map(build_text)
dataset = dataset.map(lambda example: {"labels": label2id[example[target_column]]})
dataset = dataset.cast_column("labels", ClassLabel(num_classes=len(label_names), names=label_names))
print(dataset["train"][0]["text"])

Casting the dataset: 100%|██████████| 1401862/1401862 [00:03<00:00, 430894.87 examples/s]


Свитер вязанный осень-зима 2017 Очень хороший свитер


In [10]:
train_test = dataset["train"].train_test_split(test_size=0.10, seed=SEED, stratify_by_column="labels")
val_test = train_test["test"].train_test_split(test_size=0.5, seed=SEED, stratify_by_column="labels")
train_dataset = train_test["train"]
val_dataset = val_test["train"]
test_dataset = val_test["test"]
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1261675
Validation: 70093
Test: 70094


In [11]:
def show_distribution(split, name):
    counts = Counter(split["labels"])
    print(f"\n{name}")
    for label_id, count in sorted(counts.items()):
        print(f"{label_names[label_id]}: {count}")

show_distribution(train_dataset, "TRAIN")
show_distribution(val_dataset, "VALIDATION")
show_distribution(test_dataset, "TEST")


TRAIN
Автотовары: 44962
Аксессуары: 171608
Аксессуары и принадлежности: 1871
Бытовая техника: 22323
Бытовая химия и личная гигиена: 1695
Виды спорта: 13936
Гигиена и подгузники: 606
Дача, сад и огород: 6629
Детская одежда: 37804
Детские товары: 61852
Детский спорт и активный отдых: 338
Женская одежда: 1694
Женские аксессуары: 13830
Здоровье: 5186
Зоотовары: 30730
Книги: 8655
Компьютерная техника: 875
Красота: 129667
Наушники и аудиотехника: 185
Обувь: 12025
Одежда: 142914
Отдых на природе, рыбалка и охота: 7477
Офисная техника: 680
Продукты питания: 1922
Ручной инструмент и оснастка: 1780
Смартфоны и телефоны: 35260
Спорт и отдых: 37566
Строительство и ремонт: 53356
Текстиль: 1391
Техника для дома: 2628
Техника для кухни: 683
Товары для дома: 123033
Товары для кухни: 26104
Товары для праздников: 13689
Уход за волосами: 5725
Хобби и творчество: 115673
Хозяйственные товары: 3091
Электроинструменты: 1369
Электроника: 120863

VALIDATION
Автотовары: 2498
Аксессуары: 9534
Аксессуары и прина

## 3. Tokenization

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_model_dataset = train_dataset.map(tokenize, batched=True)
val_model_dataset = val_dataset.map(tokenize, batched=True)
test_model_dataset = test_dataset.map(tokenize, batched=True)
model_columns = ["input_ids", "attention_mask", "labels"]

def keep_model_columns(split):
    split = split.remove_columns([c for c in split.column_names if c not in model_columns])
    split.set_format("torch", columns=model_columns)
    return split

train_model_dataset = keep_model_columns(train_model_dataset)
val_model_dataset = keep_model_columns(val_model_dataset)
test_model_dataset = keep_model_columns(test_model_dataset)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_model_dataset.column_names)

c:\Users\alive\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alive\.cache\huggingface\hub\models--cointegrated--rubert-tiny2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 70094/70094 [00:13<00:00, 5341.14 examples/s]

['labels', 'input_ids', 'attention_mask']


In [13]:
sample = train_model_dataset.select(range(5000))
lengths = [len(ids) for ids in sample["input_ids"]]
print("Min:", min(lengths))
print("Mean:", sum(lengths)/len(lengths))
print("Max:", max(lengths))
print("At max length:", sum(length >= MAX_LENGTH for length in lengths))
print("Share at max length:", sum(length >= MAX_LENGTH for length in lengths)/len(lengths))

Min: 4
Mean: 114.4854
Max: 256
At max length: 516
Share at max length: 0.1032


## Вывод

После фильтрации остаётся 1 401 862 объекта с 39 классами. Используем split 90/5/5 и `max_length=256`; padding выполняется динамически